# 🐉 BDH (Baby Dragon Hatchling) Training in Google Colab

This notebook clones the **BDH** repository and runs end-to-end model training, real-time logging with TensorBoard, and text generation.

> **Note:** Make sure you have enabled a GPU accelerator (**Runtime > Change runtime type > T4 GPU** or better).

## 0. Install UV

In [ ]:
!pip install -q uv

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Clone Repository and Install Dependencies

In [ ]:
# Clone the repository if not already present
import os
if not os.path.exists("BDH"):
    !git clone https://github.com/eberlful/BDH.git

%cd BDH

# Sync dependencies with uv
!uv sync

## 3. Launch TensorBoard (Optional Live Monitoring)
Run this cell before or during training to track loss curves and metrics live.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 4. Train a Model

Select one of the configurations below, or customize parameters using `--set path.to.key=value`.

### Option A: Tiny Shakespeare (Fast Baseline)

In [ ]:
# Validate configuration
!uv run python main.py validate configs/tiny_shakespeare.yaml

# Start training
!uv run python main.py train configs/tiny_shakespeare.yaml --set trainer.max_epochs=10

### Option B: BDH-CQ (In-Context Recurrent Memory & Latent Workspace)

In [ ]:
!uv run python main.py train configs/bdh_cq.yaml \
  --set trainer.max_epochs=5 \
  --set model.params.latent_reasoning_steps=4 \
  --set model.params.loss_schedule=ramp

### Option C: Sudoku Chain-of-Thought Reasoning

In [ ]:
!uv run python main.py train configs/sudoku_cot.yaml \
  --set trainer.max_epochs=5 \
  --set data.params.reasoning_mode=full

## 5. Generate Predictions from Best Checkpoint

### Option A: Text Generation (Tiny Shakespeare / BDH-CQ)
Finds the most recent text training run and generates continuous text from a prompt.

In [ ]:
import glob
import os

# Find latest run directory
runs = sorted(glob.glob("runs/*"), key=os.path.getmtime)
if runs:
    latest_run = runs[-1]
    print(f"Using latest run directory: {latest_run}")
    !uv run python main.py generate {latest_run} "To be or not to be" --max-tokens 100
else:
    print("No runs found in runs/ directory. Run a training session first.")

### Option B: Sudoku Solving & Reasoning Generation (Sudoku / Sudoku CoT)
Loads a trained Sudoku model (Option C or `bdh_sudoku.yaml`) to solve a 9x9 puzzle and generate step-by-step reasoning.

In [ ]:
import glob
import os
import random
import yaml
from src.data.data import _generate_solved_board
from src.data.sudoku_cot import format_sudoku_grid

# Locate the most recent Sudoku run
sudoku_run = None
for r in sorted(glob.glob("runs/*"), key=os.path.getmtime, reverse=True):
    cfg_path = os.path.join(r, "config.yaml")
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            cfg = yaml.safe_load(f) or {}
        if "sudoku" in cfg.get("data", {}).get("name", ""):
            sudoku_run = r
            break

if not sudoku_run:
    runs = sorted(glob.glob("runs/*"), key=os.path.getmtime)
    if runs:
        sudoku_run = runs[-1]

if sudoku_run:
    print(f"Using Sudoku run directory: {sudoku_run}\n")
    
    # 1. Generate or define an 81-digit test puzzle (0 indicates empty cells)
    rng = random.Random(42)
    solved = list(_generate_solved_board(rng))
    puzzle = list(solved)
    # Mask 51 cells leaving 30 clues
    for idx in rng.sample(range(81), 51):
        puzzle[idx] = 0
    
    puzzle_str = "".join(map(str, puzzle))
    
    print("Input Sudoku Puzzle:")
    print(format_sudoku_grid(puzzle))
    print("\n" + "=" * 50)
    print("Generating Solution / CoT Steps with BDH...")
    print("=" * 50 + "\n")
    
    # 2. Run generation CLI (auto-formats 81-digit string into puzzle prompt)
    !uv run python main.py generate {sudoku_run} "{puzzle_str}" --max-tokens 512
else:
    print("No runs found in runs/ directory. Train a Sudoku model first (Section 4, Option C).")

## 6. (Optional) Save Checkpoints to Google Drive
Mount Google Drive to persist your runs across Colab sessions.

In [ ]:
# Uncomment to mount Google Drive and back up runs
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/BDH_runs"
# !cp -r runs/* "/content/drive/MyDrive/BDH_runs/"